# Estudiante: Jean Carlos Burgos
# Carrera: Ingeniería en Ciencia de Datos e Inteligencia Artificial



In [17]:
!pip install simpy

In [18]:
import simpy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st

# Fijamos semilla para reproducibilidad
np.random.seed(42)
random.seed(42)
plt.style.use('ggplot')

In [19]:
# 1. CONFIGURACIÓN DE PARÁMETROS OPERATIVOS
TASA_LLEGADAS_LAMBDA = 30.0   # 30 peticiones por minuto
TASA_SERVICIO_MU = 10.0       # 10 imágenes por minuto por nodo GPU
NODOS_GPU_C = 4                # Número de servidores distribuidos
STOCK_INICIAL_TOKENS = 500    # Créditos iniciales de nube
CANT_RECARGA_Q = 400          # Cantidad fija a pedir al proveedor de nube
LEAD_TIME_MINUTOS = 2         # El proveedor tarda media de 2 minutos en recargar

# Parámetros del experimento
TIEMPO_SIMULACION = 60        # Duración de la ventana de estrés (en minutos)
REPLICAS = 30                 # Cantidad de experimentos independientes

PUNTO_REORDEN_S = 50


In [20]:
# 2. DEFINICIÓN DEL ENTORNO DE INFRAESTRUCTURA DE IA (DES)
class InfraestructuraAPI:
    def __init__(self, env, num_gpus, tokens_ini, punto_reorden, cant_recarga, lead_time):
        self.env = env
        # Recurso: Nodos de cómputo GPU
        self.nodos_gpu = simpy.Resource(env, capacity=num_gpus)
        # Contenedor: Stock de créditos de nube (Tokens)
        self.creditos_nube = simpy.Container(env, init=tokens_ini, capacity=5000)

        self.punto_reorden = punto_reorden
        self.cant_recarga = cant_recarga
        self.lead_time = lead_time

        # Estado lógico de pasarela de pago / recarga
        self.recargas_pendientes = 0

        # Listas de métricas operacionales
        self.latencias_cola = []
        self.registro_creditos = []
        self.registro_tiempo = []
        self.peticiones_exitosas = 0
        self.predicciones_fallidas = 0

    def solicitar_recarga_proveedor(self):
        """Simula el retraso de red y procesamiento de la recarga de créditos (Lead Time)."""
        self.recargas_pendientes += 1

        # Simulación de Lead Time con distribución normal (media=2, desv_est=0.25 para estabilidad)
        tiempo_procesamiento_pago = max(0.5, random.normalvariate(self.lead_time, 0.25))
        yield self.env.timeout(tiempo_procesamiento_pago)

        # Inyección de los créditos adquiridos al contenedor del sistema
        yield self.creditos_nube.put(self.cant_recarga)
        self.recargas_pendientes -= 1

    def monitorear_creditos(self):
        """Monitoreo continuo de la disponibilidad financiera (Política de revisión continua s, Q)."""
        while True:
            self.registro_creditos.append(self.creditos_nube.level)
            self.registro_tiempo.append(self.env.now)

            # Condición crítica de reorden financiero
            if self.creditos_nube.level <= self.punto_reorden and self.recargas_pendientes == 0:
                self.env.process(self.solicitar_recarga_proveedor())

            yield self.env.timeout(0.1) # Mayor resolución de monitoreo en minutos

def peticion_usuario(env, id_peticion, infraestructura, mu):
    """Representa el ciclo de vida de una petición HTTP entrante a la API de ML."""
    momento_arribo = env.now

    # Asignación y encolamiento en los nodos GPU
    with infraestructura.nodos_gpu.request() as asignacion_gpu:
        yield asignacion_gpu

        # Cálculo de la latencia en cola (Wq)
        latencia = env.now - momento_arribo
        infraestructura.latencias_cola.append(latencia)

        # Procesamiento e inferencia de la red neuronal (Modelo de servicio Exponencial)
        tiempo_inferencia = random.expovariate(mu)
        yield env.timeout(tiempo_inferencia)

        # Consumo de recursos de negocio: validación de tokens/créditos
        if infraestructura.creditos_nube.level > 0:
            yield infraestructura.creditos_nube.get(1) # Cada inferencia consume 1 token
            infraestructura.peticiones_exitosas += 1
        else:
            infraestructura.predicciones_fallidas += 1

def generador_trafico(env, infraestructura, lam, mu):
    """Representa la tasa de solicitudes de usuarios concurrentes (Proceso de Poisson)."""
    peticion_id = 0
    while True:
        # Intervalo entre arribos determinísticamente estocástico
        yield env.timeout(random.expovariate(lam))
        peticion_id += 1
        env.process(peticion_usuario(env, peticion_id, infraestructura, mu))

In [21]:
# 3. EVALUACIÓN ESTADÍSTICA E INFERENCIA (30 RÉPLICAS)
def calcular_ic_95(datos):
    n = len(datos)
    media = np.mean(datos)
    error_estandar = st.sem(datos)
    h = error_estandar * st.t.ppf((1 + 0.95) / 2., n - 1)
    return media, media - h, media + h

def ejecutar_experimento_operacional(replicas_n, tiempo_limite):
    metricas_wq = []
    metricas_fallas = []

    for r in range(replicas_n):
        env = simpy.Environment()
        api_ml = InfraestructuraAPI(env, NODOS_GPU_C, STOCK_INICIAL_TOKENS, PUNTO_REORDEN_S, CANT_RECARGA_Q, LEAD_TIME_MINUTOS)

        # Orquestación de procesos de eventos discretos
        env.process(generador_trafico(env, api_ml, TASA_LLEGADAS_LAMBDA, TASA_SERVICIO_MU))
        env.process(api_ml.monitorear_creditos())
        env.run(until=tiempo_limite)

        # Convertimos latencia de cola a segundos para una mejor interpretación de APIs HTTP
        latencia_promedio_segundos = np.mean(api_ml.latencias_cola) * 60
        metricas_wq.append(latencia_promedio_segundos)
        metricas_fallas.append(api_ml.predicciones_fallidas)

    # Inferencia estadística
    media_wq, ic_inf, ic_sup = calcular_ic_95(metricas_wq)
    promedio_fallas = np.mean(metricas_fallas)
    utilizacion_teorica = TASA_LLEGADAS_LAMBDA / (NODOS_GPU_C * TASA_SERVICIO_MU)

    print("======================================================================")
    print(f"      RESULTADOS OPERACIONALES DEL SISTEMA DE IA ({replicas_n} RÉPLICAS) ")
    print("======================================================================")
    print(f"-> Latencia Media en Cola (Wq):            {media_wq:.4f} segundos")
    print(f"-> Intervalo de Confianza al 95%:        [{ic_inf:.4f}, {ic_sup:.4f}] segundos")
    print(f"-> Promedio de Predicciones Fallidas:    {promedio_fallas:.2f} solicitudes")
    print(f"-> Utilización Teórica del Clúster (p): {utilizacion_teorica:.1%} (Estable)")
    print("======================================================================")

# Ejecución del pipeline de simulación
ejecutar_experimento_operacional(replicas_n=REPLICAS, tiempo_limite=TIEMPO_SIMULACION)

      RESULTADOS OPERACIONALES DEL SISTEMA DE IA (30 RÉPLICAS) 
-> Latencia Media en Cola (Wq):            3.2009 segundos
-> Intervalo de Confianza al 95%:        [2.7572, 3.6446] segundos
-> Promedio de Predicciones Fallidas:    44.20 solicitudes
-> Utilización Teórica del Clúster (p): 75.0% (Estable)


## Pregunta: ¿Por qué el sistema arroja predicciones fallidas a pesar de tener una utilización de hardware estable (rho = 75%)?

El sistema presenta fallas en las predicciones a pesar de contar con un clúster de hardware matemáticamente estable debido a un desacoplamiento crítico entre la capacidad de procesamiento físico y las reglas lógicas del negocio. Con una tasa de llegadas de $\lambda = 30$ peticiones por minuto y una capacidad de servicio conjunta de $c \cdot \mu = 40$ imágenes por minuto, la utilización teórica de los nodos GPU se sitúa en un $75\%$ (p = 0.75). Este porcentaje demuestra que la infraestructura de cómputo cuenta con la holgura suficiente para procesar la cola de solicitudes HTTP sin riesgo de saturación o colapso del servidor.


El verdadero cuello de botella se origina en la gestión financiera de la plataforma: la política de inventario continua establece el Punto de Reorden en un nivel crítico de $s = 50$ créditos (Tokens), mientras que el ritmo de la hora pico consume un promedio de 30 tokens por minuto. Dado que el proveedor de nube tarda un tiempo de espera (Lead Time) de 2 minutos en procesar la pasarela de pago e inyectar la recarga, la demanda esperada durante ese retraso logístico es de 60 tokens.
Al ser el punto de reorden original (s = 50) inferior a la necesidad real de consumo en tránsito (60), la API agota por completo sus recursos financieros a cero antes de que llegue el nuevo lote de suministro, provocando que las peticiones procesadas a tiempo por las GPUs terminen fallando a nivel lógico por falta de fondos.

# Modificar el código encontrando y configurando el valor exacto del Punto de Reorden (s) necesario para asegurar que las predicciones fallidas lleguen a cero absoluto.

CODIGO MODIFICADO

In [22]:
import simpy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st

# Fijamos semilla para reproducibilidad
np.random.seed(42)
random.seed(42)
plt.style.use('ggplot')

In [23]:
# 1. CONFIGURACIÓN DE PARÁMETROS OPERATIVOS
TASA_LLEGADAS_LAMBDA = 30.0   # 30 peticiones por minuto
TASA_SERVICIO_MU = 10.0       # 10 imágenes por minuto por nodo GPU
NODOS_GPU_C = 4                # Número de servidores distribuidos
STOCK_INICIAL_TOKENS = 500    # Créditos iniciales de nube
CANT_RECARGA_Q = 400          # Cantidad fija a pedir al proveedor de nube
LEAD_TIME_MINUTOS = 2         # El proveedor tarda media de 2 minutos en recargar

# Parámetros del experimento
TIEMPO_SIMULACION = 60        # Duración de la ventana de estrés (en minutos)
REPLICAS = 30                 # Cantidad de experimentos independientes

PUNTO_REORDEN_S = 111

In [24]:
# 2. DEFINICIÓN DEL ENTORNO DE INFRAESTRUCTURA DE IA (DES)
class InfraestructuraAPI:
    def __init__(self, env, num_gpus, tokens_ini, punto_reorden, cant_recarga, lead_time):
        self.env = env
        # Recurso: Nodos de cómputo GPU
        self.nodos_gpu = simpy.Resource(env, capacity=num_gpus)
        # Contenedor: Stock de créditos de nube (Tokens)
        self.creditos_nube = simpy.Container(env, init=tokens_ini, capacity=5000)

        self.punto_reorden = punto_reorden
        self.cant_recarga = cant_recarga
        self.lead_time = lead_time

        # Estado lógico de pasarela de pago / recarga
        self.recargas_pendientes = 0

        # Listas de métricas operacionales
        self.latencias_cola = []
        self.registro_creditos = []
        self.registro_tiempo = []
        self.peticiones_exitosas = 0
        self.predicciones_fallidas = 0

    def solicitar_recarga_proveedor(self):
        """Simula el retraso de red y procesamiento de la recarga de créditos (Lead Time)."""
        self.recargas_pendientes += 1

        # Simulación de Lead Time con distribución normal (media=2, desv_est=0.25 para estabilidad)
        tiempo_procesamiento_pago = max(0.5, random.normalvariate(self.lead_time, 0.25))
        yield self.env.timeout(tiempo_procesamiento_pago)

        # Inyección de los créditos adquiridos al contenedor del sistema
        yield self.creditos_nube.put(self.cant_recarga)
        self.recargas_pendientes -= 1

    def monitorear_creditos(self):
        """Monitoreo continuo de la disponibilidad financiera (Política de revisión continua s, Q)."""
        while True:
            self.registro_creditos.append(self.creditos_nube.level)
            self.registro_tiempo.append(self.env.now)

            # Condición crítica de reorden financiero
            if self.creditos_nube.level <= self.punto_reorden and self.recargas_pendientes == 0:
                self.env.process(self.solicitar_recarga_proveedor())

            yield self.env.timeout(0.1) # Mayor resolución de monitoreo en minutos

def peticion_usuario(env, id_peticion, infraestructura, mu):
    """Representa el ciclo de vida de una petición HTTP entrante a la API de ML."""
    momento_arribo = env.now

    # Asignación y encolamiento en los nodos GPU
    with infraestructura.nodos_gpu.request() as asignacion_gpu:
        yield asignacion_gpu

        # Cálculo de la latencia en cola (Wq)
        latencia = env.now - momento_arribo
        infraestructura.latencias_cola.append(latencia)

        # Procesamiento e inferencia de la red neuronal (Modelo de servicio Exponencial)
        tiempo_inferencia = random.expovariate(mu)
        yield env.timeout(tiempo_inferencia)

        # Consumo de recursos de negocio: validación de tokens/créditos
        if infraestructura.creditos_nube.level > 0:
            yield infraestructura.creditos_nube.get(1) # Cada inferencia consume 1 token
            infraestructura.peticiones_exitosas += 1
        else:
            infraestructura.predicciones_fallidas += 1

def generador_trafico(env, infraestructura, lam, mu):
    """Representa la tasa de solicitudes de usuarios concurrentes (Proceso de Poisson)."""
    peticion_id = 0
    while True:
        # Intervalo entre arribos determinísticamente estocástico
        yield env.timeout(random.expovariate(lam))
        peticion_id += 1
        env.process(peticion_usuario(env, peticion_id, infraestructura, mu))

In [25]:
# 3. EVALUACIÓN ESTADÍSTICA E INFERENCIA (30 RÉPLICAS)
def calcular_ic_95(datos):
    n = len(datos)
    media = np.mean(datos)
    error_estandar = st.sem(datos)
    h = error_estandar * st.t.ppf((1 + 0.95) / 2., n - 1)
    return media, media - h, media + h

def ejecutar_experimento_operacional(replicas_n, tiempo_limite):
    metricas_wq = []
    metricas_fallas = []

    for r in range(replicas_n):
        env = simpy.Environment()
        api_ml = InfraestructuraAPI(env, NODOS_GPU_C, STOCK_INICIAL_TOKENS, PUNTO_REORDEN_S, CANT_RECARGA_Q, LEAD_TIME_MINUTOS)

        # Orquestación de procesos de eventos discretos
        env.process(generador_trafico(env, api_ml, TASA_LLEGADAS_LAMBDA, TASA_SERVICIO_MU))
        env.process(api_ml.monitorear_creditos())
        env.run(until=tiempo_limite)

        # Convertimos latencia de cola a segundos para una mejor interpretación de APIs HTTP
        latencia_promedio_segundos = np.mean(api_ml.latencias_cola) * 60
        metricas_wq.append(latencia_promedio_segundos)
        metricas_fallas.append(api_ml.predicciones_fallidas)

    # Inferencia estadística
    media_wq, ic_inf, ic_sup = calcular_ic_95(metricas_wq)
    promedio_fallas = np.mean(metricas_fallas)
    utilizacion_teorica = TASA_LLEGADAS_LAMBDA / (NODOS_GPU_C * TASA_SERVICIO_MU)

    print("======================================================================")
    print(f"      RESULTADOS OPERACIONALES DEL SISTEMA DE IA ({replicas_n} RÉPLICAS) ")
    print("======================================================================")
    print(f"-> Latencia Media en Cola (Wq):            {media_wq:.4f} segundos")
    print(f"-> Intervalo de Confianza al 95%:        [{ic_inf:.4f}, {ic_sup:.4f}] segundos")
    print(f"-> Promedio de Predicciones Fallidas:    {promedio_fallas:.2f} solicitudes")
    print(f"-> Utilización Teórica del Clúster (p): {utilizacion_teorica:.1%} (Estable)")
    print("======================================================================")

# Ejecución del pipeline de simulación
ejecutar_experimento_operacional(replicas_n=REPLICAS, tiempo_limite=TIEMPO_SIMULACION)

      RESULTADOS OPERACIONALES DEL SISTEMA DE IA (30 RÉPLICAS) 
-> Latencia Media en Cola (Wq):            3.2899 segundos
-> Intervalo de Confianza al 95%:        [2.9321, 3.6477] segundos
-> Promedio de Predicciones Fallidas:    0.00 solicitudes
-> Utilización Teórica del Clúster (p): 75.0% (Estable)
